# Qwen 14B AWQ matched recovery fallback smoke — 30 requests

1. In Colab, choose **Runtime → Change runtime type → L4 GPU**.
2. Upload the INPUT `qwen14b_matched_recovery_fallback_smoke_v1.zip` from
   `artifacts/local_development_audits/recovery_matched_fallback_smoke_preparation_v1/`.
3. Run the execution cell **once**. Enter the current quota/compute-unit note,
   or explicitly `not displayed` if the provider does not show remaining quota.
4. Run DOWNLOAD after execution finishes, including a recorded failure.
   Save the OUTPUT `uqroute_qwen14b_matched_recovery_fallback_smoke_*.zip` and attach it in chat.
5. After saving the output, **disconnect and delete the GPU runtime**.

This user-run execution cell authorizes at most **30 new fallback responses**
for this development engineering smoke only: five eligible clean groups, all six
fault-feedback types per group. Initial outputs are reused, not regenerated.
No new small generation, reference scoring, final threshold or held-out evaluation.
All 30 prior fault contexts are retained without correctness/uncertainty filtering.
No later small answer is appended; this collects pairs, not a gated policy.
Preparation authorization/readiness flags remain false and unchanged.

The notebook installs isolated vLLM 0.30.0, Transformers 5.20.2 and Ninja, then
checks every local/server token sequence before generation. It retains the
requested temperature 0.001 and records the observed constructor clamp to 0.01,
inherited model defaults, FP16 AWQ, context 8192 and output cap 1024. No seed is added.
The public Qwen tokenizer/model needs no user authentication token.

Repository: `c3cbf9f44334547a747585c613afe34d381bd070`.
INPUT ZIP SHA256: `2e2bc38f6104c06d9a6084344a5d89b1dfcd2ddb7b3ec53791a7d9d65c3abc61`.
Manifest SHA256: `3bc0762f233ebfcad69f55bc20285235a8962b6a93417b73fab96d24094d816d`.

There are **zero automatic generation retries**. An existing execution folder
is refused; interrupted/unknown requests remain occupied. Do not reset and resend
this batch after an interruption. Download partial evidence for review first.
Setup/model compilation can take several minutes; the startup deadline is 20
minutes and the generation wall deadline is 15 minutes, with 300 seconds per
request. These are stopping limits, not estimates or a quota guarantee. The
quota note records operator information, not measured GPU minutes or money.

The server `/tokenize` checks do not generate model outputs. A recorded local
HTTP write establishes a submitted fault-feedback attempt, not server execution
or an actual external tool failure. Smoke status/usage/token alignment do not
measure recovery correctness. Dependent faults share only five canonical groups.
Raw prompts and model outputs stay in the private evidence ZIP. Credentials,
environment variables, dependency caches and model weights are not exported.
Partial export is best effort after a software interruption; forced runtime
deletion can lose evidence. Downloading the output again does not rerun inference.


## 1. Upload INPUT


In [ ]:
from pathlib import Path
import hashlib
from google.colab import files

expected = '2e2bc38f6104c06d9a6084344a5d89b1dfcd2ddb7b3ec53791a7d9d65c3abc61'
target = Path('/content/qwen14b_matched_recovery_fallback_smoke_v1.zip')
if target.is_symlink():
    raise RuntimeError('Symlink INPUT path; preserve it.')
if target.exists():
    if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise RuntimeError('Occupied INPUT path differs; preserve it for review.')
    print('Approved INPUT already available; no upload needed.')
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Select one INPUT transfer ZIP.')
    raw = next(iter(uploaded.values()))
    if hashlib.sha256(raw).hexdigest() != expected:
        raise RuntimeError('Wrong INPUT: select qwen14b_matched_recovery_fallback_smoke_v1.zip.')
    if not target.exists():
        with target.open('xb') as file:
            file.write(raw)
    if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise RuntimeError('Canonical INPUT differs; preserve it.')
    print('INPUT verified. Run execution once.')


## 2. Execute once


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import os
import shutil
import signal
import subprocess
import sys
import time

TASK = Path('/content/uqroute_qwen14b_matched_recovery_fallback_smoke_execution_v1')
if TASK.exists():
    raise RuntimeError('Execution folder exists. Preserve it; do not repeat generation.')
source = Path('/content/qwen14b_matched_recovery_fallback_smoke_v1.zip')
raw = source.read_bytes()
expected = '2e2bc38f6104c06d9a6084344a5d89b1dfcd2ddb7b3ec53791a7d9d65c3abc61'
if hashlib.sha256(raw).hexdigest() != expected:
    raise RuntimeError('INPUT fingerprint differs; nothing started.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True, check=True)
print('Allocated GPU:', gpu.stdout.strip(), flush=True)
if 'L4' not in gpu.stdout:
    raise RuntimeError('Choose an L4 GPU before starting.')
quota_note = input("Enter current Colab quota/compute units, or 'not displayed': ").strip()
if not quota_note:
    raise RuntimeError('A quota note is required; nothing started.')
TASK.mkdir(exist_ok=False)
(TASK / 'transfer.zip').write_bytes(raw)
(TASK / 'quota_note.txt').write_text(quota_note, encoding='utf-8')

CLIENT_SOURCE = '"""Pinned matched 30-request 14B fallback smoke; no fresh small calls or retries."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport sys\nimport zipfile\nfrom collections import Counter\nfrom pathlib import Path\n\nCODE = "c3cbf9f44334547a747585c613afe34d381bd070"\nZIP_SHA = "2e2bc38f6104c06d9a6084344a5d89b1dfcd2ddb7b3ec53791a7d9d65c3abc61"\nMANIFEST_SHA = "3bc0762f233ebfcad69f55bc20285235a8962b6a93417b73fab96d24094d816d"\nMODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"\nREV = "539535859b135b0244c91f3e59816150c8056698"\nPARENT_SHA = "dd91a73f5f565269f83d70773fd25fd64c82785029e3d96b139682c072b59595"\nMEMBERS = {\n    "manifest.json",\n    "requests.json",\n    "episodes.json",\n    "paired_small_records.json",\n    "ledger.py",\n}\n\n\ndef quantization_check(config):\n    quant = config.get("quantization_config", {})\n    if quant.get("quant_method", "").lower() != "awq" or quant.get("bits") != 4:\n        raise ValueError("Pinned model must describe a four-bit AWQ checkpoint")\n    return {\n        "model_id": MODEL,\n        "revision": REV,\n        "dtype": "float16",\n        "checkpoint_quantization": quant,\n        "backend": "Automatic selection by pinned vLLM; inspect server log",\n    }\n\n\ndef module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    result = importlib.util.module_from_spec(spec)\n    sys.modules[name] = result\n    spec.loader.exec_module(result)\n    return result\n\n\ndef dependencies(repo):\n    support = module(\n        "_recovery_gpu_support",\n        repo / "tools/development_audits/corrected_generation_v1/support.py",\n    )\n    # workflow imports its neighboring support module by its historical name.\n    sys.modules["support"] = support\n    workflow = module(\n        "_recovery_gpu_workflow",\n        repo / "tools/development_audits/corrected_generation_v1/workflow.py",\n    )\n    support.verify_baseline(repo)\n    return support, workflow\n\n\ndef load_plan(repo, prepared):\n    support, workflow = dependencies(repo)\n    payloads = {n: support.read(prepared / n) for n in MEMBERS}\n    if support.sha(payloads["manifest.json"]) != MANIFEST_SHA:\n        raise ValueError("Owner smoke manifest fingerprint differs")\n    manifest = support.loads(payloads["manifest.json"])\n    if manifest["payload_sha256"] != {\n        n: support.sha(b) for n, b in payloads.items() if n != "manifest.json"\n    }:\n        raise ValueError("Smoke payload inventory differs")\n    config = support.loads(support.source_bytes(repo)["docs/config/task1_single_sample_v1.json"])\n    if (\n        manifest["version"] != "uqroute-matched-recovery-fallback-smoke-1.0.0"\n        or manifest["purpose"] != "qwen14b-postfault-fallback-matched-smoke"\n        or manifest["model_settings"] != config["models"][3]\n        or manifest["server_settings"] != config["single_sample"]["server"]\n        or manifest["request_count"] != 30\n        or manifest["maximum_new_requests"] != 30\n        or manifest["fresh_small_requests"] != 0\n        or manifest["known_parent_small_slots"] != 30\n        or manifest["parent_smoke_archive_sha256"] != PARENT_SHA\n        or manifest["collection_mode"] != "all_matched_smoke_episodes_no_threshold"\n        or manifest["labels_used_for_selection"] is not False\n        or manifest["static_dispatcher_compatible"] is not False\n        or manifest["automatic_retries"] != 0\n        or any(\n            manifest[n] is not False\n            for n in ("dispatch_authorized", "held_out_access_enabled", "main_experiment_ready")\n        )\n    ):\n        raise ValueError("Bounded smoke scope/settings differ")\n    expected_ledger = support.read(\n        repo / "tools/development_audits/recovery_fallback_smoke_v1/ledger.py"\n    )\n    if payloads["ledger.py"] != expected_ledger.replace(b"\\r\\n", b"\\n"):\n        raise ValueError("Packaged ledger differs from pinned repository")\n    ledger = module("_recovery_gpu_ledger", prepared / "ledger.py")\n    rows, episodes = (\n        support.loads(payloads["requests.json"]),\n        support.loads(payloads["episodes.json"]),\n    )\n    ledger.checked_rows(rows)\n    if len(rows) != 30:\n        raise ValueError("Smoke request inventory differs")\n    indexed = {e["episode_key"]: e for e in episodes}\n    if len(indexed) != len(episodes) or len(episodes) != 30:\n        raise ValueError("Episode inventory differs")\n    groups = {}\n    for row in rows:\n        episode = indexed[row["episode_key"]]\n        if (\n            episode["split"] != "development"\n            or episode["population"] != "primary"\n            or episode["source_file"] != "clean.jsonl"\n            or episode["injection_eligible"] is not True\n            or episode["transition_injected"] is not None\n            or episode["context_messages_sha256"] != row["context_messages_sha256"]\n        ):\n            raise ValueError("Episode/context binding differs")\n        groups.setdefault((episode["benchmark"], episode["canonical_base_id"]), []).append(\n            episode["fault_type"]\n        )\n    faults = {\n        "timeout",\n        "rate_limit",\n        "auth_error",\n        "server_error",\n        "malformed_response",\n        "schema_drift",\n    }\n    if (\n        len(groups) != 5\n        or {b for b, _ in groups} != {"apibank", "bfcl_v3", "rotbench", "toolalpaca", "tooleyes"}\n        or any(len(v) != 6 or set(v) != faults for v in groups.values())\n    ):\n        raise ValueError("Five benchmark groups and all six faults required")\n    receipts = support.loads(payloads["paired_small_records.json"])\n    paired = {r["request_id"]: r for r in receipts}\n    if (\n        len(receipts) != 30\n        or len(paired) != 30\n        or set(paired) != {row["paired_small_request_id"] for row in rows}\n    ):\n        raise ValueError("Complete known parent receipt inventory required")\n    for row in rows:\n        parent = paired[row["paired_small_request_id"]]\n        if (\n            parent["episode_key"] != row["episode_key"]\n            or parent["context_messages_sha256"] != row["context_messages_sha256"]\n            or parent["http_status"] != 200\n            or parent["transition_injected"] is not True\n            or parent["evidence_status"] != "complete"\n        ):\n            raise ValueError("Known matching parent response receipt required")\n    return manifest, rows, indexed, ledger, support, workflow\n\n\ndef unpack(task):\n    import hashlib\n\n    raw = (task / "transfer.zip").read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ZIP_SHA:\n        raise ValueError("Owner smoke ZIP fingerprint differs")\n    prepared = task / "prepared"\n    if prepared.exists():\n        raise ValueError("Prepared execution input already exists")\n    with zipfile.ZipFile(task / "transfer.zip") as zipped:\n        if (\n            len(zipped.namelist()) != 5\n            or set(zipped.namelist()) != MEMBERS\n            or zipped.testzip() is not None\n        ):\n            raise ValueError("Smoke ZIP member inventory differs")\n        payloads = {n: zipped.read(n) for n in MEMBERS}\n    if hashlib.sha256(payloads["manifest.json"]).hexdigest() != MANIFEST_SHA:\n        raise ValueError("Owner smoke manifest fingerprint differs")\n    prepared.mkdir()\n    for name, content in payloads.items():\n        with (prepared / name).open("xb") as file:\n            file.write(content)\n\n\ndef server_context_check(rows, local_report, post):\n    indexed = {r["request_id"]: r for r in local_report["per_request"]}\n    if len(indexed) != len(rows) or set(indexed) != {r["request_id"] for r in rows}:\n        raise ValueError("Local tokenizer coverage differs")\n    import hashlib\n\n    result = []\n    for row in rows:\n        saved = indexed[row["request_id"]]\n        status, raw = post(\n            {\n                "model": MODEL,\n                "messages": row["request"]["messages"],\n                "add_generation_prompt": True,\n                "add_special_tokens": False,\n            }\n        )\n        reply = json.loads(raw)\n        tokens = reply.get("tokens")\n        if (\n            status != 200\n            or not isinstance(tokens, list)\n            or not tokens\n            or any(type(t) is not int for t in tokens)\n            or reply["count"] != len(tokens)\n            or len(tokens) != saved["prompt_tokens"]\n            or hashlib.sha256(json.dumps(tokens, separators=(",", ":")).encode()).hexdigest()\n            != saved["token_ids_sha256"]\n            or len(tokens) + row["request"]["max_tokens"] > 8192\n        ):\n            raise ValueError("Server/local tokenization mismatch; no generation allowed")\n        result.append(\n            {\n                "request_id": row["request_id"],\n                "prompt_tokens": len(tokens),\n                "token_ids_sha256": saved["token_ids_sha256"],\n                "response_sha256": hashlib.sha256(raw).hexdigest(),\n            }\n        )\n    return {\n        "requests": len(result),\n        "all_match": True,\n        "generation_requests_sent": 0,\n        "scope": "Server tokenizer endpoint only; not model generation",\n        "per_request": result,\n    }\n\n\ndef ledger_canonical(value):\n    return json.dumps(\n        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False\n    )\n\n\ndef binding_for(task, manifest, rows, support, workflow):\n    runtime_raw = support.read(task / "metadata/runtime.json")\n    runtime = support.loads(runtime_raw)\n    workflow.check_runtime(runtime, manifest)\n    local = support.loads(support.read(task / "metadata/tokenizer_preflight.json"))\n    server = support.loads(support.read(task / "metadata/server_tokenizer_preflight.json"))\n    if (\n        server["all_match"] is not True\n        or server["requests"] != len(rows)\n        or local["overflow_requests"] != 0\n        or local["requests"] != len(rows)\n    ):\n        raise ValueError("Complete matching runtime context checks required")\n    local_rows = {r["request_id"]: r for r in local["per_request"]}\n    server_rows = {r["request_id"]: r for r in server["per_request"]}\n    expected = {r["request_id"] for r in rows}\n    if (\n        set(local_rows) != expected\n        or set(server_rows) != expected\n        or len(local["per_request"]) != len(rows)\n        or len(server["per_request"]) != len(rows)\n        or local["tokenizer_revision"] != REV\n        or local["context_limit"] != 8192\n        or local["output_allowance"] != 1024\n    ):\n        raise ValueError("Runtime tokenizer identity/coverage differs")\n    for row in rows:\n        before, after = local_rows[row["request_id"]], server_rows[row["request_id"]]\n        if (\n            type(before["prompt_tokens"]) is not int\n            or before["prompt_tokens"] < 1\n            or before["prompt_tokens"] + 1024 > 8192\n            or any(before[k] != after[k] for k in ("prompt_tokens", "token_ids_sha256"))\n        ):\n            raise ValueError("Runtime tokenizer evidence differs")\n    return {\n        "purpose": manifest["purpose"],\n        "request_inventory_sha256": support.sha(ledger_canonical(rows).encode()),\n        "parent_smoke_archive_sha256": PARENT_SHA,\n        "known_parent_small_slots": manifest["known_parent_small_slots"],\n        "collection_mode": manifest["collection_mode"],\n        "prepared_manifest_sha256": MANIFEST_SHA,\n        "source_archive_sha256": ZIP_SHA,\n        "runtime_attestation": runtime,\n        "runtime_attestation_sha256": support.sha(runtime_raw),\n        "context_preflight_sha256": {\n            name: support.sha(support.read(task / "metadata" / name))\n            for name in ("tokenizer_preflight.json", "server_tokenizer_preflight.json")\n        },\n        "client_source_sha256": support.sha(Path(__file__).read_bytes()),\n        "automatic_retries": 0,\n        "held_out_access_enabled": False,\n        "main_experiment_ready": False,\n        "endpoint": "http://127.0.0.1:8000/v1",\n        "authorization": "Explicit user-run notebook for 30 smoke requests only",\n    }\n\n\ndef run(task, execute=False):\n    if execute is not True:\n        raise ValueError("Explicit --execute-development required")\n    manifest, rows, _, ledger, support, workflow = load_plan(task / "repo", task / "prepared")\n    binding = binding_for(task, manifest, rows, support, workflow)\n    if (task / "run").exists():\n        raise ValueError("Existing execution run; preserve it, do not repeat smoke requests")\n    status, raw = workflow.transport(binding["endpoint"], "/models", timeout=30)\n    if status != 200 or MODEL not in {m["id"] for m in support.loads(raw)["data"]}:\n        raise ValueError("Wrong local model server")\n    result = ledger.dispatch(\n        rows,\n        task / "run",\n        binding,\n        ledger.local_caller(binding["endpoint"]),\n        execute_development=True,\n    )\n    ledger.json_once(task / "metadata/dispatch_summary.json", result)\n    print(json.dumps({n: v for n, v in result.items() if n != "states"}, indent=2), flush=True)\n    if result["saved_responses"] != 30 or result["known_http_errors"] or result["unknown_slots"]:\n        raise ValueError("Incomplete smoke; preserve all slots and do not retry")\n\n\ndef replay(task):\n    manifest, rows, episodes, ledger, support, workflow = load_plan(\n        task / "repo", task / "prepared"\n    )\n    binding = binding_for(task, manifest, rows, support, workflow)\n    states = ledger.replay(rows, task / "run", binding)\n    frozen = support.methods(task / "repo")\n    records = []\n    sums, missing = {"prompt_tokens": 0, "completion_tokens": 0}, 0\n    for row in rows:\n        state = states["states"][row["request_id"]]\n        record = {\n            "request_id": row["request_id"],\n            "episode_key": row["episode_key"],\n            "context_messages_sha256": row["context_messages_sha256"],\n            "paired_small_request_id": row["paired_small_request_id"],\n            "stage": "postfault_fallback",\n            "benchmark": episodes[row["episode_key"]]["benchmark"],\n            "canonical_base_id": episodes[row["episode_key"]]["canonical_base_id"],\n            "fault_type": episodes[row["episode_key"]]["fault_type"],\n            "ledger_state": state,\n            "transition_injected": True if state.startswith("injected_") else None,\n            "official_correctness": None,\n            "validation": None,\n            "usage": None,\n            "initial_usage": episodes[row["episode_key"]].get("initial_usage"),\n        }\n        folder = task / "run/cases" / row["request_id"]\n        if state == "injected_response_saved":\n            meta = support.loads(support.read(folder / "response_metadata.json"))\n            record["http_status"] = meta["http_status"]\n            record["elapsed_seconds"] = meta["elapsed_seconds"]\n            raw = support.read(folder / "http_response.bin")\n            try:\n                response = support.loads(raw)\n                record["usage"] = response.get("usage") if isinstance(response, dict) else None\n                record["validation"] = (\n                    workflow.evaluate(response, record["benchmark"], frozen)\n                    if meta["http_status"] == 200\n                    else {"status": "http_error"}\n                )\n            except (ValueError, KeyError, TypeError, OverflowError, AttributeError) as exc:\n                record["validation"] = {\n                    "status": "validation_error",\n                    "exception_type": type(exc).__name__,\n                }\n        usage = record["usage"]\n        if isinstance(usage, dict) and all(\n            type(usage.get(n)) is int and usage[n] >= 0 for n in sums\n        ):\n            for name in sums:\n                sums[name] += usage[name]\n        else:\n            missing += 1\n        records.append(record)\n    output = task / "replay"\n    if output.exists():\n        raise ValueError("Replay output already exists; preserve it")\n    output.mkdir()\n    summary = {n: v for n, v in states.items() if n != "states"}\n    summary.update(\n        validation_status_counts=dict(\n            Counter((r["validation"] or {}).get("status", "no_saved_response") for r in records)\n        ),\n        known_usage_sums=sums,\n        missing_usage_slots=missing,\n        usage_totals=None if missing else sums,\n        fallback_requests=states["observed_injected"],\n        maximum_fallback_requests=30,\n        fresh_small_requests=0,\n        collection_mode="all_matched_smoke_episodes_no_threshold",\n        initial_generation="reused source evidence, not new calls",\n        held_out_access_enabled=False,\n        main_experiment_ready=False,\n    )\n    ledger.json_once(output / "records.json", records)\n    ledger.json_once(output / "summary.json", summary)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=("run", "replay"))\n    parser.add_argument("--task", type=Path, required=True)\n    parser.add_argument("--execute-development", action="store_true")\n    args = parser.parse_args()\n    if args.mode == "run":\n        run(args.task.resolve(), args.execute_development)\n    else:\n        replay(args.task.resolve())\n\n\nif __name__ == "__main__":\n    main()\n'
RUNNER_SOURCE = '"""Colab L4 supervisor for one pinned 30-request matched fallback smoke."""\n\nfrom __future__ import annotations\n\nimport http.client\nimport json\nimport os\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\nfrom importlib import metadata\nfrom pathlib import Path\n\nimport client\n\n\ndef export_evidence(task):\n    """Export completed or partial evidence, excluding environment/cache/credentials."""\n    contents = {}\n    for prefix in ("prepared", "metadata", "run", "replay"):\n        folder = task / prefix\n        if folder.is_symlink():\n            raise ValueError("Symlink evidence directory")\n        if folder.is_dir():\n            for path in sorted(folder.rglob("*")):\n                if path.is_symlink():\n                    raise ValueError("Symlink evidence file")\n                if path.is_file():\n                    contents[prefix + "/" + path.relative_to(folder).as_posix()] = path.read_bytes()\n    import hashlib\n\n    hashes = {name: hashlib.sha256(raw).hexdigest() for name, raw in contents.items()}\n    archive = task.parent / (\n        "uqroute_qwen14b_matched_recovery_fallback_smoke_" + uuid.uuid4().hex[:8] + ".zip"\n    )\n    with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as zipped:\n        for name, raw in contents.items():\n            zipped.writestr(name, raw)\n        zipped.writestr(\n            "metadata/export_file_hashes.json", json.dumps(hashes, sort_keys=True) + "\\n"\n        )\n    with zipfile.ZipFile(archive) as zipped:\n        if zipped.testzip() is not None or any(\n            hashlib.sha256(zipped.read(n)).hexdigest() != h for n, h in hashes.items()\n        ):\n            raise ValueError("Evidence export verification failed")\n    with (task / "export_path.txt").open("x", encoding="utf-8") as file:\n        file.write(str(archive))\n    return {\n        "archive": str(archive),\n        "sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),\n        "evidence_files": len(hashes),\n    }\n\n\ndef main(task):\n    task = task.resolve()\n    meta = task / "metadata"\n    meta.mkdir(exist_ok=False)\n    server, generation, failure = None, None, None\n    started = time.monotonic()\n\n    def write(name, value):\n        with (meta / name).open("x", encoding="utf-8") as file:\n            json.dump(value, file, indent=2, allow_nan=False)\n            file.write("\\n")\n\n    def stop(process):\n        if process is not None and process.poll() is None:\n            os.killpg(process.pid, signal.SIGTERM)\n            try:\n                process.wait(timeout=20)\n            except subprocess.TimeoutExpired:\n                os.killpg(process.pid, signal.SIGKILL)\n                process.wait(timeout=10)\n\n    def command(args, **kwargs):\n        return subprocess.run(args, check=True, **kwargs)\n\n    def post_tokenize(body):\n        connection = http.client.HTTPConnection("127.0.0.1", 8000, timeout=30)\n        try:\n            connection.request(\n                "POST", "/tokenize", json.dumps(body).encode(), {"Content-Type": "application/json"}\n            )\n            response = connection.getresponse()\n            return response.status, response.read()\n        finally:\n            connection.close()\n\n    def interrupted(signum, frame):\n        raise KeyboardInterrupt("Supervisor interrupted; preserve request states")\n\n    signal.signal(signal.SIGTERM, interrupted)\n    signal.signal(signal.SIGINT, interrupted)\n\n    try:\n        import torch\n\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA and the frozen FP16 AWQ model required")\n        device = torch.cuda.get_device_properties(0)\n        if "L4" not in device.name:\n            raise RuntimeError("Select the L4 GPU for this pinned notebook")\n        if metadata.version("vllm") != "0.30.0" or metadata.version("transformers") != "5.20.2":\n            raise RuntimeError("Pinned dependency versions differ")\n        quota = (task / "quota_note.txt").read_text().strip()\n        if not quota:\n            raise RuntimeError(\n                "Record current provider quota or explicitly note it is not displayed"\n            )\n        write(\n            "environment.json",\n            {\n                "python": sys.version,\n                "torch": metadata.version("torch"),\n                "vllm": metadata.version("vllm"),\n                "transformers": metadata.version("transformers"),\n                "torch_cuda_build": torch.version.cuda,\n                "gpu_name": device.name,\n                "gpu_memory_mib": device.total_memory // 1024**2,\n                "quota_note": quota,\n            },\n        )\n        ninja = shutil.which("ninja")\n        if not ninja:\n            raise RuntimeError("Ninja absent from isolated PATH")\n        write(\n            "ninja_check.json",\n            {\n                "path": ninja,\n                "version": command(\n                    [ninja, "--version"], capture_output=True, text=True\n                ).stdout.strip(),\n            },\n        )\n        client.unpack(task)\n        repo = task / "repo"\n        command(\n            [\n                "git",\n                "clone",\n                "--no-checkout",\n                "https://github.com/RavindraSSK/uqroute-tc.git",\n                str(repo),\n            ],\n            capture_output=True,\n            text=True,\n        )\n        command(\n            ["git", "-C", str(repo), "checkout", "--detach", client.CODE],\n            capture_output=True,\n            text=True,\n        )\n        if (\n            command(\n                ["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True\n            ).stdout.strip()\n            != client.CODE\n            or command(\n                ["git", "-C", str(repo), "status", "--porcelain"], capture_output=True, text=True\n            ).stdout\n        ):\n            raise RuntimeError("Pinned repository checkout differs")\n        manifest, rows, _, ledger, support, workflow = client.load_plan(repo, task / "prepared")\n        write(\n            "repository_validation.json",\n            {\n                "revision": client.CODE,\n                "manifest_sha256": client.MANIFEST_SHA,\n                "zip_sha256": client.ZIP_SHA,\n                "requests": len(rows),\n            },\n        )\n        from huggingface_hub import hf_hub_download\n        from transformers import AutoTokenizer\n\n        tokenizer = AutoTokenizer.from_pretrained(\n            client.MODEL, revision=client.REV, trust_remote_code=False\n        )\n        counts = []\n        for row in rows:\n            ids = tokenizer.apply_chat_template(\n                row["request"]["messages"],\n                tokenize=True,\n                add_generation_prompt=True,\n                return_dict=False,\n            )\n            if not isinstance(ids, list) or not ids or any(type(t) is not int for t in ids):\n                raise RuntimeError("Unexpected tokenizer IDs")\n            counts.append(\n                {\n                    "request_id": row["request_id"],\n                    "prompt_tokens": len(ids),\n                    "token_ids_sha256": ledger.sha(json.dumps(ids, separators=(",", ":")).encode()),\n                }\n            )\n        preflight = {\n            "requests": len(counts),\n            "prompt_tokens_max": max(r["prompt_tokens"] for r in counts),\n            "overflow_requests": sum(r["prompt_tokens"] + 1024 > 8192 for r in counts),\n            "output_allowance": 1024,\n            "context_limit": 8192,\n            "tokenizer_revision": client.REV,\n            "per_request": counts,\n        }\n        write("tokenizer_preflight.json", preflight)\n        if preflight["overflow_requests"]:\n            raise RuntimeError("Context overflow; no trimming/dropping allowed")\n        config_raw = Path(\n            hf_hub_download(client.MODEL, "config.json", revision=client.REV)\n        ).read_bytes()\n        quantization = client.quantization_check(json.loads(config_raw))\n        with (meta / "model_config.json").open("xb") as file:\n            file.write(config_raw)\n        write("quantization_check.json", quantization)\n        generation_raw = Path(\n            hf_hub_download(client.MODEL, "generation_config.json", revision=client.REV)\n        ).read_bytes()\n        with (meta / "model_generation_config.json").open("xb") as file:\n            file.write(generation_raw)\n        from vllm.entrypoints.openai.chat_completion.protocol import ChatCompletionRequest\n\n        observed = set()\n        for row in rows:\n            params = ChatCompletionRequest(**row["request"]).to_sampling_params(\n                max_tokens=1024, default_sampling_params=json.loads(generation_raw)\n            )\n            observed.add(\n                (\n                    params.temperature,\n                    params.top_p,\n                    params.top_k,\n                    params.repetition_penalty,\n                    params.max_tokens,\n                    params.n,\n                    params.seed,\n                )\n            )\n        if len(observed) != 1:\n            raise RuntimeError("Nonuniform installed sampling constructor")\n        temperature, top_p, top_k, penalty, output_cap, n, seed = next(iter(observed))\n        if temperature != 0.01 or output_cap != 1024 or n != 1 or seed is not None:\n            raise RuntimeError("Installed sampling differs from recorded static-run behavior")\n        write(\n            "sampling_constructor_check.json",\n            {\n                "requested_temperature": 0.001,\n                "constructed_temperature": temperature,\n                "top_p": top_p,\n                "top_k": top_k,\n                "repetition_penalty": penalty,\n                "max_tokens": output_cap,\n                "n": n,\n                "seed": seed,\n                "scope": ("Installed local constructor; server sampler not independently observed"),\n            },\n        )\n        with socket.socket() as probe:\n            if probe.connect_ex(("127.0.0.1", 8000)) == 0:\n                raise RuntimeError("Port 8000 occupied; preserve existing server")\n        argv = [\n            sys.executable,\n            "-m",\n            "vllm.entrypoints.openai.api_server",\n            "--model",\n            client.MODEL,\n            "--revision",\n            client.REV,\n            "--tokenizer-revision",\n            client.REV,\n            "--dtype",\n            "float16",\n            "--max-model-len",\n            "8192",\n            "--gpu-memory-utilization",\n            "0.88",\n            "--host",\n            "127.0.0.1",\n            "--port",\n            "8000",\n        ]\n        write("launch_argv.json", argv)\n        print("Loading the pinned Qwen 14B AWQ model on L4.", flush=True)\n        with (meta / "server.log").open("xb") as log:\n            server = subprocess.Popen(\n                argv, stdout=log, stderr=subprocess.STDOUT, start_new_session=True\n            )\n        startup = time.monotonic()\n        last = 0\n        while True:\n            if server.poll() is not None or time.monotonic() - startup > 1200:\n                raise RuntimeError("Server failed/startup deadline exceeded; preserve logs")\n            try:\n                status, raw = workflow.transport("http://127.0.0.1:8000/v1", "/models", timeout=5)\n                if status != 200 or client.MODEL not in {\n                    m["id"] for m in support.loads(raw)["data"]\n                }:\n                    raise RuntimeError("Unexpected local model server")\n                break\n            except (OSError, TimeoutError):\n                if time.monotonic() - last >= 30:\n                    print("Server loading/compiling; waiting.", flush=True)\n                    last = time.monotonic()\n                time.sleep(2)\n        snapshot = (meta / "server.log").read_bytes()\n        with (meta / "server_before_generation.log").open("xb") as file:\n            file.write(snapshot)\n        server_check = client.server_context_check(rows, preflight, post_tokenize)\n        write("server_tokenizer_preflight.json", server_check)\n        runtime = {\n            k: manifest["model_settings"][k]\n            for k in ("model_id", "model_revision", "tokenizer_revision", "dtype", "quantization")\n        }\n        runtime.update(\n            vllm_version=metadata.version("vllm"),\n            max_model_len=8192,\n            gpu_memory_utilization=0.88,\n            chat_template=manifest["server_settings"]["chat_template"],\n            effective_temperature=temperature,\n            gpu_name=device.name,\n            gpu_memory_mib=device.total_memory // 1024**2,\n            platform="Google Colab L4",\n            quota_note=quota,\n            launch_command_sha256=ledger.sha((meta / "launch_argv.json").read_bytes()),\n            server_log_sha256=ledger.sha(snapshot),\n        )\n        workflow.check_runtime(runtime, manifest)\n        write("runtime.json", runtime)\n        write(\n            "runtime_limits.json",\n            {\n                "sampling": "Local constructor observation, not per-request server sampler proof",\n                "model_identity": (\n                    "Pinned launch arguments; model ID endpoint does not verify weights"\n                ),\n                "quota": "Operator note; available GPU minutes and completion are not guaranteed",\n                "batch_wall_limit_seconds": 900,\n                "per_request_timeout_seconds": 300,\n                "tokenizer_requests": 30,\n                "maximum_generation_requests": 30,\n            },\n        )\n        write(\n            "execution_authorization.json",\n            {\n                "scope": "User-run notebook authorizes exactly this 30-request development smoke",\n                "preparation_dispatch_authorized": False,\n                "maximum_fallback_requests": 30,\n                "fresh_small_requests": 0,\n                "parent_smoke_archive_sha256": client.PARENT_SHA,\n                "held_out_access_enabled": False,\n                "main_experiment_ready": False,\n            },\n        )\n        print(\n            "All 30 server/local token sequences match. Generating the 30 smoke responses once.",\n            flush=True,\n        )\n        generation_start = time.monotonic()\n        with (\n            (meta / "generation_stdout.log").open("xb") as out,\n            (meta / "generation_stderr.log").open("xb") as err,\n        ):\n            generation = subprocess.Popen(\n                [\n                    sys.executable,\n                    str(task / "client.py"),\n                    "run",\n                    "--task",\n                    str(task),\n                    "--execute-development",\n                ],\n                stdout=out,\n                stderr=err,\n                start_new_session=True,\n            )\n            last = 0\n            while generation.poll() is None:\n                if time.monotonic() - generation_start > 900:\n                    stop(generation)\n                    raise RuntimeError("900-second generation wall limit; preserve unknown slots")\n                if time.monotonic() - last >= 30:\n                    slots = task / "run/cases"\n                    occupied = len(list(slots.glob("*/intent.json")))\n                    saved = len(list(slots.glob("*/response_metadata.json")))\n                    print(\n                        f"Smoke progress: {occupied}/30 intents, {saved}/30 saved responses.",\n                        flush=True,\n                    )\n                    last = time.monotonic()\n                time.sleep(2)\n        write(\n            "generation_exit.json",\n            {\n                "returncode": generation.returncode,\n                "wall_seconds": time.monotonic() - generation_start,\n            },\n        )\n        replay = subprocess.run(\n            [sys.executable, str(task / "client.py"), "replay", "--task", str(task)],\n            capture_output=True,\n            text=True,\n        )\n        write(\n            "replay_command.json",\n            {"returncode": replay.returncode, "stdout": replay.stdout, "stderr": replay.stderr},\n        )\n        print(replay.stdout, flush=True)\n        if generation.returncode or replay.returncode:\n            raise RuntimeError("Smoke dispatch/replay incomplete; preserve evidence, do not retry")\n        write(\n            "execution_status.json",\n            {\n                "status": "smoke_generation_and_replay_finished",\n                "official_scoring": "not_performed",\n                "rq3_outcomes": "not_evaluated",\n                "held_out_access_enabled": False,\n                "main_experiment_ready": False,\n            },\n        )\n    except BaseException as exc:\n        failure = type(exc).__name__ + ": " + str(exc)\n        write(\n            "execution_error.json",\n            {"error": failure, "action": "Preserve evidence; no request retries"},\n        )\n        print("STOPPED:", failure, flush=True)\n    finally:\n        for name, process in (("generation", generation), ("server", server)):\n            try:\n                stop(process)\n            except Exception as exc:\n                write(name + "_shutdown_error.json", {"exception_type": type(exc).__name__})\n        write(\n            "shutdown.json",\n            {\n                "server_stopped": server is None or server.poll() is not None,\n                "generation_stopped": generation is None or generation.poll() is not None,\n                "supervisor_wall_seconds": time.monotonic() - started,\n                "scope": "Supervisor wall time; not all GPU allocation or billing",\n            },\n        )\n        for name in ("runner.py", "client.py", "environment_install.log", "dependency_check.log"):\n            if (task / name).is_file():\n                shutil.copyfile(task / name, meta / name)\n        exported = export_evidence(task)\n        print(json.dumps(exported, indent=2), flush=True)\n        print(\n            "Server shutdown attempted. Download evidence before deleting the runtime.", flush=True\n        )\n    return 1 if failure else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main(Path(sys.argv[1])))\n'
(TASK / 'client.py').write_text(CLIENT_SOURCE, encoding='utf-8')
(TASK / 'runner.py').write_text(RUNNER_SOURCE, encoding='utf-8')
execution, install = None, None
try:
    ENV = TASK / 'venv'
    PYTHON = ENV / 'bin/python'
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(ENV)], check=True)
    print('Installing the isolated environment; model loading follows setup.', flush=True)
    with (TASK / 'environment_install.log').open('xb') as log:
        install = subprocess.Popen([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install',
                                    'pip', 'vllm==0.30.0', 'transformers==5.20.2', 'ninja'],
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        last = 0
        while install.poll() is None:
            if time.monotonic() - last >= 30:
                print('Dependency installation is running.', flush=True)
                last = time.monotonic()
            time.sleep(2)
    if install.returncode:
        raise RuntimeError('Dependency install failed; preserve the installation log.')
    env = os.environ.copy()
    env['PATH'] = str(PYTHON.parent) + os.pathsep + env.get('PATH', '')
    env['VIRTUAL_ENV'] = str(ENV)
    env.pop('PYTHONPATH', None)
    with (TASK / 'dependency_check.log').open('xb') as log:
        subprocess.run([str(PYTHON), '-m', 'pip', 'check'], env=env, stdout=log,
                       stderr=subprocess.STDOUT, check=True)
    print('Starting runtime validation and the bounded smoke.', flush=True)
    execution = subprocess.Popen([str(PYTHON), str(TASK / 'runner.py'), str(TASK)],
                                 env=env, start_new_session=True)
    execution.wait()
    print('Execution exit code:', execution.returncode, flush=True)
except BaseException as exc:
    if install is not None and install.poll() is None:
        os.killpg(install.pid, signal.SIGTERM)
        try:
            install.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(install.pid, signal.SIGKILL)
            install.wait(timeout=10)
    if execution is not None and execution.poll() is None:
        os.killpg(execution.pid, signal.SIGINT)
        try:
            execution.wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(execution.pid, signal.SIGKILL)
            execution.wait(timeout=10)
    print('Execution stopped:', type(exc).__name__, str(exc), flush=True)
finally:
    if not (TASK / 'export_path.txt').is_file():
        # Salvage setup/partial artifacts without executing the supervisor main.
        meta = TASK / 'metadata'
        meta.mkdir(exist_ok=True)
        for name in ('client.py', 'runner.py', 'environment_install.log', 'dependency_check.log'):
            if (TASK / name).is_file() and not (meta / name).exists():
                shutil.copyfile(TASK / name, meta / name)
        (meta / 'outer_salvage.txt').write_text(
            'Partial/setup evidence only. '
            'Server shutdown and dispatch completion may be unknown.\n',
            encoding='utf-8')
        sys.path.insert(0, str(TASK))
        spec = importlib.util.spec_from_file_location('_smoke_export_only', TASK / 'runner.py')
        exporter = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(exporter)
        print(exporter.export_evidence(TASK), flush=True)
    print('Download the OUTPUT evidence ZIP now; do not rerun this execution cell.', flush=True)


## 3. Download OUTPUT


In [ ]:
from pathlib import Path
from google.colab import files
task = Path('/content/uqroute_qwen14b_matched_recovery_fallback_smoke_execution_v1')
marker = task / 'export_path.txt'
if not marker.is_file():
    raise RuntimeError('No export marker. Preserve runtime/logs; do not rerun generation.')
output = Path(marker.read_text().strip())
if not output.is_file():
    raise RuntimeError('OUTPUT export missing; preserve runtime.')
print('Downloading OUTPUT:', output.name)
files.download(str(output))
